# Out-of-Sample Evaluation: Classical vs Prescriptive vs DRCC

## Experimental design

All three methods are trained on **one dataset** (off-peak season).
They are then evaluated on **two test sets** that differ in distribution from the training data:

| Test set | Distribution | Expected story |
|---|---|---|
| **Close** | Same off-peak distribution (different seed) | All methods perform similarly — no distribution shift |
| **Far** | Peak season — high demand unseen in training | Classical & Prescriptive under-commit → load shedding; DRCC is robust |

**Why this design?**  In practice, you train on the data you have (historical).
The question is what happens at **deployment** when conditions shift.
DRCC's Wasserstein ambiguity set (radius ρ_z) hedges against exactly this shift:
the policy is optimised to remain feasible even if the true distribution
is ρ_z-far from the training distribution in Wasserstein distance.

**Features are demand forecasts.** The policy sees only the day-ahead demand forecast
`z` (features). The realised demand `d` (forecast `z = d + forecast error`) is unknown when the commitment is
decided; it enters training only through the power-balance / line constraints (and the oracle
labels of the classical model) and is used out of sample only to dispatch and score the schedule.

**Out-of-sample evaluation**: learned policy (on forecasts) → fixed commitment b → solve dispatch LP
against the **realised** demand
with load-shedding slack (penalised at 1000 $/MWh) → decompose into operation + load-shedding cost.


## 1  Imports & System Data


In [ ]:
import numpy as np
import pandas as pd
import gurobipy as grb
import matplotlib.pyplot as plt
import time


In [ ]:
np.random.seed(42)

# ── Dimensions ───────────────────────────────────────────────────────────────
n_bus, n_g, n_t, n_load, n_line = 3, 3, 24, 3, 3

# ── Generator data ────────────────────────────────────────────────────────────
gen_bus  = [0, 1, 2]
pmax = pd.DataFrame([0.55, 0.40, 0.20])
pmin = pd.DataFrame([0.05, 0.05, 0.00])
rs   = pd.DataFrame([0.30, 0.25, 0.40])
UT   = pd.DataFrame([2, 2, 1])
DT   = pd.DataFrame([2, 2, 1])
c_op = pd.DataFrame([15.0, 40.0,  8.0])   # Gen1 is mid-cost
c_on = pd.DataFrame([ 1.0,  5.0,  1.0])
c_st = pd.DataFrame([ 5.0, 20.0,  3.0])
c_ls = 1000.0   # load-shedding penalty $/MWh

load_bus  = [0, 1, 2]
load_base = [0.50, 0.30, 0.20]
load      = pd.DataFrame(load_base)
total_load_scale = sum(load_base)   # = 1.0

lines    = [(0,1),(0,2),(1,2)]
b_line   = 10.0
fmax_val = 0.60
fmax     = pd.DataFrame([fmax_val]*n_line)

# ── PTDF ──────────────────────────────────────────────────────────────────────
B = np.zeros((n_bus,n_bus))
for (i,j) in lines:
    B[i,i]+=b_line; B[j,j]+=b_line; B[i,j]-=b_line; B[j,i]-=b_line
B_red_inv = np.linalg.inv(B[1:,1:])
H_ptdf = np.zeros((n_line,n_bus))
for idx,(fr,to) in enumerate(lines):
    for n in range(1,n_bus):
        th=np.zeros(n_bus); th[1:]=B_red_inv[:,n-1]
        H_ptdf[idx,n] = b_line*(th[fr]-th[to])
Hg = np.column_stack([H_ptdf[:,gen_bus[j]] for j in range(n_g)])
Hl = np.column_stack([H_ptdf[:,load_bus[l]] for l in range(n_load)])
beta = Hl @ np.array(load_base)
line_dual_norm = np.sum(np.abs(Hg),axis=1) + np.abs(beta)
bal_dual_norm  = total_load_scale + n_g   # = 4.0

# ── Base load profile ─────────────────────────────────────────────────────────
base_profile = np.array([
    0.30,0.28,0.26,0.25,0.27,0.32,
    0.40,0.50,0.65,0.74,0.78,0.82,
    0.84,0.85,0.83,0.82,0.85,0.88,
    0.90,0.88,0.82,0.72,0.55,0.42,
])
threshold = float(pmax.iloc[0].item()) + float(pmax.iloc[2].item())   # 0.75
hours = np.arange(n_t)
print(f'System ready. Gen capacities: {pmax.values.flatten()}, '
      f'Gen1-commitment threshold={threshold}')


## 2  Scenario Sets

The demand scenarios below are the **realised demand** `d` (identical to the previous version). The policy
features are the **day-ahead forecasts** `z = d + ε`, with ε ~ N(0, SIGMA_FC²) per hour; `d` is used for the balance/line constraints in training,
for the oracle labels of the classical model, for the hindsight benchmark and for the out-of-sample dispatch.

| Set | Demand regime | Demand scales | Purpose |
|---|---|---|---|
| **Training** | Moderate, peak occasionally above threshold | HIGH=0.83, MED=0.62, LOW=0.45 | What the policy is trained on |
| **Validation** | Same as training | HIGH=0.83, MED=0.62, LOW=0.45 | Hyperparameter tuning (held-out) |
| **Close test** | Same as training | HIGH=0.83, MED=0.62, LOW=0.45 | Deployment under known conditions |
| **Far test** | Near-peak (mild distribution shift) | HIGH=0.93, MED=0.85, LOW=0.78 | Deployment under distribution shift |

**Key property of the training data:**  
Peak training demand ≈ 0.83 × 0.90 ± noise ≈ 0.72–0.77 p.u., i.e. it crosses the Gen1 threshold (0.75 p.u.)
on a few peak hours of the high-demand days. The oracle commits Gen1 only in these rare cases, so the classical
labels for Gen1 are almost always 0 and the regression learns *never commit Gen1*, while the prescriptive model must
cover the realised demand in every training scenario and learns to commit Gen1 when the forecast is high.  

**Why the far test is challenging:**  
Far test HIGH scenarios reach ≈ 0.87 p.u. (Gen1 always needed).  
Far test MED scenarios straddle the threshold ≈ 0.75 p.u. (Gen1 sometimes needed).  
Far test LOW scenarios stay at ≈ 0.70 p.u. (Gen1 not needed).  
Classical and Prescriptive never commit Gen1 → load shedding on HIGH and MED scenarios.  
**DRCC** uses a Wasserstein ball of radius ρ_z ≈ WD(train, far test) to cover the threshold
and learns a policy that commits Gen1 whenever z approaches 0.75 p.u.


In [ ]:
from scipy.stats import wasserstein_distance

n_train, n_test = 30, 30

SIGMA_FC = 0.02   # std. dev. of the day-ahead forecast error (p.u.)

def make_scenarios(rng, scale_high=1.20, scale_med=0.82, scale_low=0.60,
                   n_high=10, n_med=10, n_low=10):
    """Generate REALISED demand scenarios as scaled + noisy copies of base_profile (as before)."""
    zh = np.clip(base_profile*scale_high + rng.uniform(-0.03, 0.03, (n_high, n_t)), 0.10, 0.99)
    zm = np.clip(base_profile*scale_med  + rng.uniform(-0.05, 0.05, (n_med,  n_t)), 0.10, 0.99)
    zl = np.clip(base_profile*scale_low  + rng.uniform(-0.03, 0.03, (n_low,  n_t)), 0.10, 0.99)
    return np.vstack([zh, zm, zl])

OFF_PEAK  = dict(scale_high=0.83,   # was 0.78: Gen1 now needed on a few peak training hours
 scale_med=0.62, scale_low=0.45,
                 n_high=10, n_med=10, n_low=10)
# Far test: near-peak, mild distribution shift — demand straddles the threshold
#   HIGH: z_peak ≈ 0.87 (Gen1 always needed)
#   MED:  z_peak ≈ 0.75 (Gen1 sometimes needed)
#   LOW:  z_peak ≈ 0.70 (Gen1 not needed)
NEAR_PEAK = dict(scale_high=0.93, scale_med=0.85, scale_low=0.78,
                 n_high=10, n_med=10, n_low=10)

# ── Realised demand: exactly the same scenarios as before ─────────────────────
d_train      = make_scenarios(np.random.default_rng(42),  **OFF_PEAK)
d_val        = make_scenarios(np.random.default_rng(55),  **OFF_PEAK)
d_test_close = make_scenarios(np.random.default_rng(7),   **OFF_PEAK)
d_test_far   = make_scenarios(np.random.default_rng(99),  **NEAR_PEAK)

# ── Features: day-ahead FORECAST = realised demand + forecast error ───────────
# The policy only sees z (forecast); d (realised) is used in the training constraints,
# for the oracle labels / hindsight, and for the out-of-sample dispatch.
def forecast(d, seed):
    eps_fc = SIGMA_FC * np.random.default_rng(seed).standard_normal(d.shape)
    return np.clip(d + eps_fc, 0.05, 0.99)

z_train      = forecast(d_train,      1042)
z_val        = forecast(d_val,        1055)
z_test_close = forecast(d_test_close, 1007)
z_test_far   = forecast(d_test_far,   1099)

n_k_tr = n_train
n_k_te = n_test

# ── Wasserstein distances: training → each test set ───────────────────────────
wd_close_h = [wasserstein_distance(z_train[:, t], z_test_close[:, t]) for t in range(n_t)]
wd_far_h   = [wasserstein_distance(z_train[:, t], z_test_far[:,   t]) for t in range(n_t)]
print(f'Mean WD  training → close test : {np.mean(wd_close_h):.4f}')
print(f'Mean WD  training → far   test : {np.mean(wd_far_h):.4f}')
print(f'Far/close WD ratio             : {np.mean(wd_far_h)/np.mean(wd_close_h):.1f}x')
print(f'Forecast error: sigma = {SIGMA_FC} p.u., forecast MAE = {np.mean(np.abs(d_train-z_train)):.4f} p.u.')
print(f'\nTraining  peak forecast range : [{z_train.max(axis=1).min():.3f}, {z_train.max(axis=1).max():.3f}]')
print(f'Training  peak realised range : [{d_train.max(axis=1).min():.3f}, {d_train.max(axis=1).max():.3f}]')
print(f'Far test  peak forecast range : [{z_test_far.max(axis=1).min():.3f}, {z_test_far.max(axis=1).max():.3f}]')
print(f'Gen0+Gen2 capacity (threshold) : {threshold:.3f}')


In [ ]:
# ── Per-hour histogram: training vs close test vs far test ───────────────────
fig, axes = plt.subplots(4, 6, figsize=(20, 11), sharey=False)
axes = axes.flatten()
for t in range(n_t):
    ax = axes[t]
    ax.hist(z_train[:,      t], bins=10, alpha=0.55, color='steelblue',
            density=True, label='Training dataset')
    ax.hist(z_test_close[:, t], bins=10, alpha=0.55, color='forestgreen',
            density=True, label='Close test dataset')
    ax.hist(z_test_far[:,   t], bins=10, alpha=0.55, color='firebrick',
            density=True, label='Far test dataset')
    ax.axvline(threshold, color='k', ls='--', lw=1.2)
    ax.set_title(f'Hour {t+1}\nWD_cl={wd_close_h[t]:.3f}  WD_fa={wd_far_h[t]:.3f}',
                 fontsize=7.5)
    ax.set_xlabel('Demand forecast (p.u.)', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.3)
    if t == 0: ax.legend(fontsize=7)
plt.suptitle('Per-Hour Demand Forecast Distributions (features)', fontsize=12)
# plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig('demand_distributions_oos.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Hour 18 demand histogram: training vs close test vs far test ─────────────
t = 17  # hour 18 (0-indexed)
fig, ax = plt.subplots(figsize=(5, 3.5))

ax.hist(z_train[:,      t], bins=10, alpha=0.55, color='steelblue',
        density=True, label='Training dataset')
ax.hist(z_test_close[:, t], bins=10, alpha=0.55, color='forestgreen',
        density=True, label='Close test dataset')
ax.hist(z_test_far[:,   t], bins=10, alpha=0.55, color='firebrick',
        density=True, label='Far test dataset')

ax.set_xlabel('Demand forecast (p.u.)', fontsize=14, fontweight='bold')
ax.set_ylabel('Distribution', fontsize=14, fontweight='bold')

ax.tick_params(axis='both', labelsize=11)
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight('bold')

leg = ax.legend(fontsize=9, frameon=True, handlelength=1.5,
                borderpad=0.4, labelspacing=0.4)
for text in leg.get_texts():
    text.set_fontweight('bold')
ax.grid(True, color='#bdbdbd', linewidth=0.8, alpha=0.8)

plt.tight_layout()
plt.savefig('demand_hour18.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Time-series view: training, close test, far test ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)

for k in range(n_train):
    axes[0].plot(hours, z_train[k],      color='steelblue',   alpha=0.25, lw=0.8)
axes[0].axhline(threshold, color='k', ls='--', lw=1.5)
axes[0].set_ylabel('Demand forecast'); axes[0].grid(alpha=0.3)
axes[0].set_title(f'Training — sub-threshold moderate demand (avg WD to far test = {np.mean(wd_far_h):.4f})',
                  fontsize=11, color='steelblue')

for k in range(n_test):
    axes[1].plot(hours, z_test_close[k], color='forestgreen', alpha=0.25, lw=0.8)
axes[1].axhline(threshold, color='k', ls='--', lw=1.5)
axes[1].set_ylabel('Demand forecast'); axes[1].grid(alpha=0.3)
axes[1].set_title(f'Close test — same sub-threshold distribution (avg WD = {np.mean(wd_close_h):.4f})',
                  fontsize=11, color='forestgreen')

for k in range(n_test):
    axes[2].plot(hours, z_test_far[k],   color='firebrick',   alpha=0.25, lw=0.8)
axes[2].axhline(threshold, color='k', ls='--', lw=1.5, label=f'Gen1 threshold {threshold}')
axes[2].set_xlabel('Hour'); axes[2].set_ylabel('Demand forecast'); axes[2].grid(alpha=0.3)
axes[2].set_title(f'Far test — near-peak demand, straddling Gen1 threshold',
                  fontsize=11, color='firebrick')

axes[2].legend(fontsize=9)

fig.suptitle('One training set, two test sets\n'
             'Far test has high demand never seen in training — Gen1 needed but never learned',
             fontsize=12)
plt.tight_layout(); plt.show()


## 3  Training: Classical (Two-Stage)

Trained on the **off-peak training set**.  
The oracle (Stage 1 deterministic UC, realised demand) commits Gen1 only on the few training hours where demand
exceeds the 0.75 p.u. threshold.  
Stage 2 L1 regression on the forecasts fits the majority label → Gen1 (almost) never committed.  
This causes load shedding on high-demand hours of the close test and on the far test.


In [ ]:
def solve_det_UC(z_k):
    m=grb.Model(); m.Params.OutputFlag=0
    b=m.addVars(n_g,n_t,vtype=grb.GRB.BINARY)
    u=m.addVars(n_g,n_t,vtype=grb.GRB.BINARY)
    p=m.addVars(n_g,n_t,lb=0.0)
    for j in range(n_g):
        for t in range(n_t):
            m.addConstr(p[j,t]<=float(pmax.iloc[j].item())*b[j,t])
            m.addConstr(p[j,t]>=float(pmin.iloc[j].item())*b[j,t])
            if t>=1:
                m.addConstr(u[j,t]>=b[j,t]-b[j,t-1])
                m.addConstr(p[j,t]-p[j,t-1]<=float(rs.iloc[j].item()))
                m.addConstr(p[j,t-1]-p[j,t]<=float(rs.iloc[j].item()))
    for t in range(1,n_t):
        for j in range(n_g):
            ut_j=int(UT.iloc[j].item()); dt_j=int(DT.iloc[j].item())
            for tau in range(t,min(t+ut_j-1,n_t)):
                m.addConstr(b[j,tau]>=b[j,t]-b[j,t-1])
            for tau in range(t,min(t+dt_j-1,n_t)):
                m.addConstr(1-b[j,tau]>=b[j,t-1]-b[j,t])
    for t in range(n_t):
        m.addConstr(grb.quicksum(p[j,t] for j in range(n_g))>=total_load_scale*z_k[t])
    for ll in range(n_line):
        for t in range(n_t):
            fl=grb.quicksum(float(Hg[ll,j])*p[j,t] for j in range(n_g))
            m.addConstr( fl-float(beta[ll])*z_k[t]<=fmax_val)
            m.addConstr(-fl+float(beta[ll])*z_k[t]<=fmax_val)
    m.setObjective(grb.quicksum(
        float(c_op.iloc[j].item())*p[j,t]+float(c_on.iloc[j].item())*b[j,t]
        +float(c_st.iloc[j].item())*u[j,t]
        for j in range(n_g) for t in range(n_t)),grb.GRB.MINIMIZE)
    m.optimize()
    if m.Status in(2,11):
        return np.array([[b[j,t].X for t in range(n_t)] for j in range(n_g)])
    return np.zeros((n_g,n_t))

def train_classical(z_tr, d_tr, c_zeta=12.0):
    """Labels: oracle UC for the REALISED demand d_tr. Regression: on the FORECAST z_tr."""
    n_k = len(z_tr)
    print('  Stage 1: oracle labels...')
    b_star = np.array([solve_det_UC(d_tr[k]) for k in range(n_k)])
    print(f'  Gen1 commit rate in oracle labels: {b_star[:,1,:].mean():.3f}')
    print('  Stage 2: L1 regression...')
    m2=grb.Model(); m2.Params.OutputFlag=0
    w =m2.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    cn=m2.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    zt=m2.addVars(n_k,n_g,n_t,lb=0.0)
    for j in range(n_g):
        for t in range(n_t):
            for k in range(n_k):
                pi = w[j,t]*z_tr[k,t] + cn[j,t]
                m2.addConstr(zt[k,j,t] >=  pi - b_star[k,j,t])
                m2.addConstr(zt[k,j,t] >= -pi + b_star[k,j,t])
    m2.setObjective((c_zeta/n_k)*grb.quicksum(
        zt[k,j,t] for k in range(n_k) for j in range(n_g) for t in range(n_t)),
        grb.GRB.MINIMIZE)
    m2.optimize()
    return (np.array([[w [j,t].X for t in range(n_t)] for j in range(n_g)]),
            np.array([[cn[j,t].X for t in range(n_t)] for j in range(n_g)]))

print('Training classical on off-peak data...')
t0=time.time()
w_cl, con_cl = train_classical(z_train, d_train)
print(f'  done ({time.time()-t0:.1f}s)')


## 4  Training: Nominal Prescriptive


In [ ]:
def train_prescriptive(z_tr, d_tr, c_zeta=12.0):
    """Policy on the FORECAST z_tr; balance and line constraints on the REALISED demand d_tr."""
    n_k=len(z_tr)
    m=grb.Model('nom'); m.Params.OutputFlag=0
    m.Params.MIPGap=0.02; m.Params.TimeLimit=300
    b=m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    u=m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    w=m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    con=m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    p=m.addVars(n_k,n_g,n_t,lb=0.0)
    zt=m.addVars(n_k,n_g,n_t,lb=0.0)
    m.update()
    for j in range(n_g):
        for t in range(n_t):
            for k in range(n_k):
                pi=w[j,t]*z_tr[k,t]+con[j,t]
                m.addConstr(pi<=0.5+b[k,j,t])
                m.addConstr(pi>=b[k,j,t]-0.5)
                m.addConstr(zt[k,j,t]>= pi-b[k,j,t])
                m.addConstr(zt[k,j,t]>=-pi+b[k,j,t])
                m.addConstr(pi>=0.0); m.addConstr(pi<=1.0)
    for k in range(n_k):
        for j in range(n_g):
            for t in range(n_t):
                m.addConstr(p[k,j,t]<=float(pmax.iloc[j].item())*b[k,j,t])
                m.addConstr(p[k,j,t]>=float(pmin.iloc[j].item())*b[k,j,t])
                if t>=1:
                    m.addConstr(u[k,j,t]>=b[k,j,t]-b[k,j,t-1])
                    m.addConstr(p[k,j,t]-p[k,j,t-1]<=float(rs.iloc[j].item()))
                    m.addConstr(p[k,j,t-1]-p[k,j,t]<=float(rs.iloc[j].item()))
    for t in range(1,n_t):
        for j in range(n_g):
            for k in range(n_k):
                ut_j=int(UT.iloc[j].item()); dt_j=int(DT.iloc[j].item())
                for tau in range(t,min(t+ut_j-1,n_t)):
                    m.addConstr(b[k,j,tau]>=b[k,j,t]-b[k,j,t-1])
                for tau in range(t,min(t+dt_j-1,n_t)):
                    m.addConstr(1-b[k,j,tau]>=b[k,j,t-1]-b[k,j,t])
    for t in range(n_t):
        for k in range(n_k):
            m.addConstr(grb.quicksum(p[k,j,t] for j in range(n_g))>=total_load_scale*d_tr[k,t])
    for i in range(n_line):
        for t in range(n_t):
            for k in range(n_k):
                gf=grb.quicksum(float(Hg[i,j])*p[k,j,t] for j in range(n_g))
                m.addConstr( gf-float(beta[i])*d_tr[k,t]<=fmax_val)
                m.addConstr(-gf+float(beta[i])*d_tr[k,t]<=fmax_val)
    m.setObjective((1.0/n_k)*(
        grb.quicksum(float(c_op.iloc[j].item())*p[k,j,t]
                     +float(c_on.iloc[j].item())*b[k,j,t]
                     +float(c_st.iloc[j].item())*u[k,j,t]
                     for k in range(n_k) for j in range(n_g) for t in range(n_t))
        +grb.quicksum(c_zeta*zt[k,j,t]
                      for k in range(n_k) for j in range(n_g) for t in range(n_t))
    ),grb.GRB.MINIMIZE)
    m.optimize()
    if m.SolCount == 0 or m.Status not in (2, 11):
        print(f'    [skip] status={m.Status}'); return None, None
    try:
        w_v=np.array([[w[j,t].X   for t in range(n_t)] for j in range(n_g)])
        c_v=np.array([[con[j,t].X for t in range(n_t)] for j in range(n_g)])
    except AttributeError:
        return None, None
    return w_v, c_v

print('Training prescriptive on off-peak data...')
t0=time.time()
w_pr, con_pr = train_prescriptive(z_train, d_train)
print(f'  done ({time.time()-t0:.1f}s)')


## 5  Helper Functions

Defined here — before DRCC tuning — so `tune_drcc` can call them.


In [ ]:
def predict_commitment(w_v, c_v, z_te):
    """Apply linear policy to FORECAST scenarios → binary commitment array (n_k, n_g, n_t)."""
    n_k=len(z_te)
    b_pred=np.zeros((n_k,n_g,n_t))
    for k in range(n_k):
        for j in range(n_g):
            for t in range(n_t):
                pi = w_v[j,t]*z_te[k,t] + c_v[j,t]
                b_pred[k,j,t] = 1.0 if pi >= 0.5 else 0.0
    return b_pred


In [ ]:
def eval_oos(b_pred, z_te):
    # z_te here is the REALISED demand (the schedule b_pred was computed from forecasts)
    """Fix commitment b_pred, solve dispatch LP with load-shedding slack.
    Returns {'op','on','st','ls','total'}."""
    n_k=len(z_te)
    m=grb.Model(); m.Params.OutputFlag=0
    u =m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    p =m.addVars(n_k,n_g,n_t,lb=0.0)
    ls=m.addVars(n_k,n_t,lb=0.0)
    m.update()
    for k in range(n_k):
        for j in range(n_g):
            for t in range(n_t):
                bval=float(b_pred[k,j,t])
                m.addConstr(p[k,j,t]<=float(pmax.iloc[j].item())*bval)
                m.addConstr(p[k,j,t]>=float(pmin.iloc[j].item())*bval)
                if t>=1:
                    m.addConstr(u[k,j,t]>=bval-float(b_pred[k,j,t-1]))
                    m.addConstr(p[k,j,t]-p[k,j,t-1]<=float(rs.iloc[j].item()))
                    m.addConstr(p[k,j,t-1]-p[k,j,t]<=float(rs.iloc[j].item()))
        for t in range(n_t):
            gen=grb.quicksum(p[k,j,t] for j in range(n_g))
            m.addConstr(gen+ls[k,t]>=total_load_scale*float(z_te[k,t]))
    cost_op = grb.quicksum(float(c_op.iloc[j].item())*p[k,j,t]
                           for k in range(n_k) for j in range(n_g) for t in range(n_t))
    cost_on = grb.quicksum(float(c_on.iloc[j].item())*float(b_pred[k,j,t])
                           for k in range(n_k) for j in range(n_g) for t in range(n_t))
    cost_st = grb.quicksum(float(c_st.iloc[j].item())*u[k,j,t]
                           for k in range(n_k) for j in range(n_g) for t in range(n_t))
    cost_ls = grb.quicksum(c_ls*ls[k,t] for k in range(n_k) for t in range(n_t))
    m.setObjective(cost_op+cost_on+cost_st+cost_ls, grb.GRB.MINIMIZE)
    m.optimize()
    op  = sum(float(c_op.iloc[j].item())*p[k,j,t].X
              for k in range(n_k) for j in range(n_g) for t in range(n_t))
    on  = float(cost_on.getValue())
    st  = sum(float(c_st.iloc[j].item())*u[k,j,t].X
              for k in range(n_k) for j in range(n_g) for t in range(n_t))
    ls_ = c_ls*sum(ls[k,t].X for k in range(n_k) for t in range(n_t))
    return {'op':op, 'on':on, 'st':st, 'ls':ls_, 'total':op+on+st+ls_}


## 6  Training: DRCC Prescriptive

DRCC adds a **Wasserstein chance constraint** to the nominal prescriptive formulation.
The policy must remain feasible for any demand distribution within Wasserstein distance ρ_z
of the training distribution, with violation probability at most ε.

### Why ρ_z cannot be tuned by cross-validation here

The validation set has the **same sub-threshold distribution** as training. All three methods
achieve zero load-shedding on it, so a grid search over ρ_z always picks the **smallest**
value (cheapest). DRCC would then behave identically to nominal prescriptive on the far test.

### How ρ_z is set

The Wasserstein ball must be large enough to **reach the Gen1 commitment threshold (0.75 p.u.)**
from the training distribution (max demand ≈ 0.73 p.u.).
We set ρ_z to cover the mean per-hour WD from training to the far test, plus a 30% safety margin,
and scan downward until the MIP is feasible.  Only **ε** is tuned by grid search.

$$\rho_z \;\geq\; W_1(\text{training},\; \text{far test}) \;\approx\; 0.75 - \max(z_{\text{train}})$$


**Computation.** The big-M constants in `train_drcc` are set to their tightest valid values and valid
cardinality cuts (at most ⌊εS⌋ violated samples per chance constraint) are added. This is an equivalent
formulation: same feasible set and optimal solutions, but much faster MIP solves. The tuned (ρ_z, ε) are
saved to `drcc_tuning.json` and reused in later runs.


**Time limits.** A run that stops at the time limit is kept if its MIP gap is at most 10% (`max_gap`);
otherwise it is discarded. Only runs without any solution count as infeasible. The final DRCC model is
re-solved with a 1800 s limit.


In [ ]:
def train_drcc(z_tr, d_tr, c_zeta=12.0, rho_z=0.10, rho_d=0.001, eps=0.10,
               time_limit=600, max_gap=0.10):
    """Policy-consistency DRCC on the FORECAST z_tr; balance/line DRCC on the REALISED demand d_tr."""
    n_k=len(z_tr); S=n_k
    z_max=float(z_tr.max()); p_tot=float(pmax.values.sum())
    # Big-M constants are set to their tightest valid values (equivalent model, faster solves):
    #   policy slacks lie in [-0.5, 1.5] (0 <= pi <= 1, b binary); balance slack in [-d, p_tot];
    #   line slacks are bounded using the PTDF and generator limits (gmin/gmax below).

    m=grb.Model('drcc'); m.Params.OutputFlag=0
    m.Params.MIPGap=0.03; m.Params.TimeLimit=time_limit

    b   =m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    u   =m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    w   =m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    con =m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    p   =m.addVars(n_k,n_g,n_t,lb=0.0)
    zt  =m.addVars(n_k,n_g,n_t,lb=0.0)
    that_p=m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    q_p   =m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    r_p   =m.addVars(n_k,n_g,n_t,lb=0.0)
    aw_p  =m.addVars(n_g,n_t,lb=0.0)
    that_m=m.addVars(n_g,n_t,lb=-grb.GRB.INFINITY)
    q_m   =m.addVars(n_k,n_g,n_t,vtype=grb.GRB.BINARY)
    r_m   =m.addVars(n_k,n_g,n_t,lb=0.0)
    aw_m  =m.addVars(n_g,n_t,lb=0.0)
    that_bal=m.addVars(n_t,lb=-grb.GRB.INFINITY)
    q_bal   =m.addVars(n_k,n_t,vtype=grb.GRB.BINARY)
    r_bal   =m.addVars(n_k,n_t,lb=0.0)
    that_lp=m.addVars(n_line,n_t,lb=-grb.GRB.INFINITY)
    q_lp   =m.addVars(n_line,n_k,n_t,vtype=grb.GRB.BINARY)
    r_lp   =m.addVars(n_line,n_k,n_t,lb=0.0)
    that_lm=m.addVars(n_line,n_t,lb=-grb.GRB.INFINITY)
    q_lm   =m.addVars(n_line,n_k,n_t,vtype=grb.GRB.BINARY)
    r_lm   =m.addVars(n_line,n_k,n_t,lb=0.0)
    m.update()

    for j in range(n_g):
        for t in range(n_t):
            m.addConstr(aw_p[j,t]>= w[j,t]); m.addConstr(aw_p[j,t]>=-w[j,t])
            m.addConstr(aw_m[j,t]>= w[j,t]); m.addConstr(aw_m[j,t]>=-w[j,t])
            m.addConstr(eps*S*that_p[j,t]-grb.quicksum(r_p[k,j,t] for k in range(n_k))>=rho_z*S*(aw_p[j,t]+1.0))
            m.addConstr(eps*S*that_m[j,t]-grb.quicksum(r_m[k,j,t] for k in range(n_k))>=rho_z*S*(aw_m[j,t]+1.0))
            for k in range(n_k):
                zkt=float(z_tr[k,t])
                m.addConstr(-w[j,t]*zkt-con[j,t]+b[k,j,t]+0.5+0.5*q_p[k,j,t]>=that_p[j,t]-r_p[k,j,t])
                m.addConstr(1.5*(1-q_p[k,j,t])>=that_p[j,t]-r_p[k,j,t])
                m.addConstr( w[j,t]*zkt+con[j,t]-b[k,j,t]+0.5+0.5*q_m[k,j,t]>=that_m[j,t]-r_m[k,j,t])
                m.addConstr(1.5*(1-q_m[k,j,t])>=that_m[j,t]-r_m[k,j,t])
                m.addConstr(zt[k,j,t]>= w[j,t]*zkt+con[j,t]-b[k,j,t])
                m.addConstr(zt[k,j,t]>=-w[j,t]*zkt-con[j,t]+b[k,j,t])
                m.addConstr(w[j,t]*zkt+con[j,t]>=0.0)
                m.addConstr(w[j,t]*zkt+con[j,t]<=1.0)
    for k in range(n_k):
        for j in range(n_g):
            for t in range(n_t):
                m.addConstr(p[k,j,t]<=float(pmax.iloc[j].item())*b[k,j,t])
                m.addConstr(p[k,j,t]>=float(pmin.iloc[j].item())*b[k,j,t])
                if t>=1:
                    m.addConstr(u[k,j,t]>=b[k,j,t]-b[k,j,t-1])
                    m.addConstr(p[k,j,t]-p[k,j,t-1]<=float(rs.iloc[j].item()))
                    m.addConstr(p[k,j,t-1]-p[k,j,t]<=float(rs.iloc[j].item()))
    for t in range(1,n_t):
        for j in range(n_g):
            for k in range(n_k):
                ut_j=int(UT.iloc[j].item()); dt_j=int(DT.iloc[j].item())
                for tau in range(t,min(t+ut_j-1,n_t)):
                    m.addConstr(b[k,j,tau]>=b[k,j,t]-b[k,j,t-1])
                for tau in range(t,min(t+dt_j-1,n_t)):
                    m.addConstr(1-b[k,j,tau]>=b[k,j,t-1]-b[k,j,t])
    for t in range(n_t):
        m.addConstr(eps*S*that_bal[t]-grb.quicksum(r_bal[k,t] for k in range(n_k))>=rho_d*S*bal_dual_norm)
        for k in range(n_k):
            Gkt=grb.quicksum(p[k,g,t] for g in range(n_g))
            m.addConstr(Gkt-total_load_scale*float(d_tr[k,t])+float(d_tr[k,t])*q_bal[k,t]>=that_bal[t]-r_bal[k,t])
            m.addConstr(p_tot*(1-q_bal[k,t])>=that_bal[t]-r_bal[k,t])
    for i in range(n_line):
        dn=float(line_dual_norm[i]); bi=float(beta[i]); hg=[float(Hg[i,j]) for j in range(n_g)]
        gmax=sum(max(hg[j],0.0)*float(pmax.iloc[j].item()) for j in range(n_g)); gmin=sum(min(hg[j],0.0)*float(pmax.iloc[j].item()) for j in range(n_g))
        for t in range(n_t):
            m.addConstr(eps*S*that_lp[i,t]-grb.quicksum(r_lp[i,k,t] for k in range(n_k))>=rho_d*S*dn)
            m.addConstr(eps*S*that_lm[i,t]-grb.quicksum(r_lm[i,k,t] for k in range(n_k))>=rho_d*S*dn)
            for k in range(n_k):
                dkt=float(d_tr[k,t])
                gf=grb.quicksum(hg[j]*p[k,j,t] for j in range(n_g))
                m.addConstr(-gf+bi*dkt+fmax_val+max(0.0,gmax-bi*dkt-fmax_val)*q_lp[i,k,t]>=that_lp[i,t]-r_lp[i,k,t])
                m.addConstr((-gmin+bi*dkt+fmax_val)*(1-q_lp[i,k,t])>=that_lp[i,t]-r_lp[i,k,t])
                m.addConstr( gf-bi*dkt+fmax_val+max(0.0,-gmin+bi*dkt-fmax_val)*q_lm[i,k,t]>=that_lm[i,t]-r_lm[i,k,t])
                m.addConstr((gmax-bi*dkt+fmax_val)*(1-q_lm[i,k,t])>=that_lm[i,t]-r_lm[i,k,t])
    # valid cuts: in each individual DRCC fewer than eps*S samples can be "violated" (q=1)
    kmax = int(np.floor(eps*S))
    for j in range(n_g):
        for t in range(n_t):
            m.addConstr(grb.quicksum(q_p[k,j,t] for k in range(n_k)) <= kmax)
            m.addConstr(grb.quicksum(q_m[k,j,t] for k in range(n_k)) <= kmax)
    for t in range(n_t):
        m.addConstr(grb.quicksum(q_bal[k,t] for k in range(n_k)) <= kmax)
        for i in range(n_line):
            m.addConstr(grb.quicksum(q_lp[i,k,t] for k in range(n_k)) <= kmax)
            m.addConstr(grb.quicksum(q_lm[i,k,t] for k in range(n_k)) <= kmax)
    m.setObjective((1.0/S)*(
        grb.quicksum(float(c_op.iloc[j].item())*p[k,j,t]
                     +float(c_on.iloc[j].item())*b[k,j,t]
                     +float(c_st.iloc[j].item())*u[k,j,t]
                     for k in range(n_k) for j in range(n_g) for t in range(n_t))
        +grb.quicksum(c_zeta*zt[k,j,t]
                      for k in range(n_k) for j in range(n_g) for t in range(n_t))
    ),grb.GRB.MINIMIZE)
    m.optimize()
    if m.SolCount == 0:                                  # no solution: infeasible
        print(f'    [skip] status={m.Status}, SolCount={m.SolCount}')
        return None, None
    if m.Status not in (2, 11):                          # time limit with a solution
        if m.MIPGap > max_gap:                           # solution too far from optimal: discard
            print(f'    [skip] status={m.Status}, gap={100*m.MIPGap:.1f}% > {100*max_gap:.0f}%')
            return None, None
        print(f'    [time limit] solution accepted, gap={100*m.MIPGap:.1f}%')
    try:
        w_v=np.array([[w[j,t].X   for t in range(n_t)] for j in range(n_g)])
        c_v=np.array([[con[j,t].X for t in range(n_t)] for j in range(n_g)])
    except AttributeError:
        return None, None
    return w_v, c_v


In [ ]:
# ── ρ_z: data-driven, scan down from WD-based upper bound ───────────────────
# ρ_z must be large enough for the Wasserstein ball to reach peak demand, but
# if it is too large the training MIP becomes infeasible (the ball covers
# physically-impossible scenarios with no load-shedding slack in training).
# Strategy: start at 1.3×mean_WD_far and halve until at least one ε is feasible.
rho_d = 0.001
mean_wd_far = float(np.mean(wd_far_h))
print(f'Mean per-hour WD  training → far test : {mean_wd_far:.4f}')

eps_candidates = [0.10, 0.15, 0.20, 0.30]

def tune_drcc(z_tr, d_tr, z_val_set, d_val_set, rho_d):
    """
    Joint search over ρ_z and ε.
    Scans ρ_z from largest (1.3×WD_far) downward in steps of 0.5×.
    At the first ρ_z where any ε is feasible, picks the best ε by val cost.
    Returns (best_eps, best_rho_z, df_results).
    """
    # 1.3x and 0.9x WD were infeasible / never solved in earlier runs; add them back to search the full grid
    rho_z_factors = [0.6, 0.4, 0.25, 0.15]
    best_cost = np.inf; best_eps = None; best_rho_z = None; all_rows = []
    for factor in rho_z_factors:
        rho_z = mean_wd_far * factor
        print(f'\n--- ρ_z = {factor}×WD = {rho_z:.4f} ---')
        print(f'  {"eps":>5}  {"op+on+st":>12}  {"load-shed":>10}  {"total":>12}  sec')
        found_feasible = False
        local_best = np.inf; local_best_eps = None; local_best_wc = None
        for eps_c in eps_candidates:
            t0 = time.time()
            w_v, c_v = train_drcc(z_tr, d_tr, rho_z=rho_z, rho_d=rho_d, eps=eps_c)
            elapsed = time.time() - t0
            if w_v is None:
                print(f'  {eps_c:>5.2f}  [INFEASIBLE]  ({elapsed:.0f}s)')
                all_rows.append(dict(rho_z=rho_z, eps=eps_c, op=np.nan, ls=np.nan, total=np.nan))
                continue
            found_feasible = True
            b_v = predict_commitment(w_v, c_v, z_val_set)
            res = eval_oos(b_v, d_val_set)
            tot = res['total']
            mark = ' <--' if tot < local_best else ''
            print(f'  {eps_c:>5.2f}  {res["op"]+res["on"]+res["st"]:>12,.0f}  '
                  f'{res["ls"]:>10,.0f}  {tot:>12,.0f}  {elapsed:.0f}s{mark}')
            all_rows.append(dict(rho_z=rho_z, eps=eps_c,
                                 op=res['op']+res['on']+res['st'],
                                 ls=res['ls'], total=tot))
            if tot < local_best:
                local_best = tot; local_best_eps = eps_c; local_best_wc = (w_v, c_v)
        if found_feasible:
            best_rho_z = rho_z; best_eps = local_best_eps; best_cost = local_best
            np.savez('drcc_tuned_policy.npz', w=local_best_wc[0], c=local_best_wc[1])   # keep the selected policy
            print(f'  => Feasible at ρ_z={rho_z:.4f}, best ε={best_eps}, val={best_cost:,.0f}')
            break  # stop at largest feasible ρ_z
    if best_eps is None:
        raise RuntimeError('DRCC infeasible for all (ρ_z, ε) combinations tried.')
    return best_eps, best_rho_z, pd.DataFrame(all_rows)

# ── Tune once per dataset, then reuse the saved hyperparameters ──────────────
# Delete drcc_tuning.json (or set TUNE = True) after changing the data or the model.
import json, os
TUNE = not os.path.exists('drcc_tuning.json')

if TUNE:
    best_eps, rho_z_drcc, df_tune = tune_drcc(z_train, d_train, z_val, d_val, rho_d)
    json.dump({'eps': best_eps, 'rho_z': rho_z_drcc}, open('drcc_tuning.json', 'w'))
    df_tune.to_csv('drcc_tuning_table.csv', index=False)
else:
    sel = json.load(open('drcc_tuning.json'))
    best_eps, rho_z_drcc = sel['eps'], sel['rho_z']
    print('Loaded tuned hyperparameters from drcc_tuning.json')
print(f'\nSelected: ρ_z={rho_z_drcc:.4f}  ε={best_eps}')


In [ ]:
import os
print(f'Final DRCC training: eps={best_eps}, rho_z={rho_z_drcc:.4f}')
t0=time.time()
# final model: longer time limit for a better solution
w_dr, con_dr = train_drcc(z_train, d_train, rho_z=rho_z_drcc, rho_d=rho_d, eps=best_eps, time_limit=1800)
if w_dr is None:
    # the re-solve returned no accepted solution: use the policy selected during tuning
    if os.path.exists('drcc_tuned_policy.npz'):
        pol = np.load('drcc_tuned_policy.npz'); w_dr, con_dr = pol['w'], pol['c']
        print('  Final re-solve not accepted -> using the policy selected during tuning (drcc_tuned_policy.npz)')
    else:
        raise RuntimeError('No DRCC policy available: delete drcc_tuning.json and re-run the tuning cell.')
print(f'  done ({time.time()-t0:.1f}s)')


## 7  Apply Policies to Both Test Sets


In [ ]:
# Policies are applied to the FORECASTS z; hindsight uses the REALISED demand d
# Close test (same off-peak distribution)
b_cl_close = predict_commitment(w_cl, con_cl, z_test_close)
b_pr_close = predict_commitment(w_pr, con_pr, z_test_close)
b_dr_close = predict_commitment(w_dr, con_dr, z_test_close)

# Far test (peak season — unseen in training)
b_cl_far   = predict_commitment(w_cl, con_cl, z_test_far)
b_pr_far   = predict_commitment(w_pr, con_pr, z_test_far)
b_dr_far   = predict_commitment(w_dr, con_dr, z_test_far)

# Hindsight: oracle commitment for each test set independently
print('Computing hindsight (oracle) commitments...')
b_hint_close = np.array([solve_det_UC(d_test_close[k]) for k in range(n_k_te)])
b_hint_far   = np.array([solve_det_UC(d_test_far[k])   for k in range(n_k_te)])
print('Done.')

# Gen1 commit rates
peak = list(range(17, 21))
print(f'\n{"Method":<18} {"Gen1 rate (close test)":>22} {"Gen1 rate (far test)":>22}')
for lbl, bc, bf in [
        ('Classical',   b_cl_close, b_cl_far),
        ('Prescriptive',b_pr_close, b_pr_far),
        ('DRCC',        b_dr_close, b_dr_far),
        ('Hindsight-cl',b_hint_close, b_hint_close),
        ('Hindsight-fa',b_hint_far,   b_hint_far),
]:
    rc = bc[:,1,:][:,peak].mean(); rf = bf[:,1,:][:,peak].mean()
    print(f'{lbl:<18} {rc:>22.3f} {rf:>22.3f}')


## 8  Out-of-Sample Cost Evaluation

Evaluate each policy on each test set by solving the dispatch LP with load-shedding slack.


In [ ]:
print('Evaluating all policies out-of-sample (dispatch against REALISED demand)...')
results = {}
eval_cases = [
    ('Classical-close',    b_cl_close,   d_test_close),
    ('Prescriptive-close', b_pr_close,   d_test_close),
    ('DRCC-close',         b_dr_close,   d_test_close),
    ('Hindsight-close',    b_hint_close, d_test_close),
    ('Classical-far',      b_cl_far,     d_test_far),
    ('Prescriptive-far',   b_pr_far,     d_test_far),
    ('DRCC-far',           b_dr_far,     d_test_far),
    ('Hindsight-far',      b_hint_far,   d_test_far),
]
for lbl, bv, zt in eval_cases:
    t0=time.time()
    results[lbl] = eval_oos(bv, zt)
    r=results[lbl]
    print(f'{lbl:<22}  op={r["op"]:>10.0f}  on={r["on"]:>8.0f}  '
          f'st={r["st"]:>7.0f}  ls={r["ls"]:>10.0f}  '
          f'total={r["total"]:>12.0f}  ({time.time()-t0:.1f}s)')


## 9  Results

**Expected story:**

| | Close test (same distribution) | Far test (distribution shift) |
|---|---|---|
| **Operation cost** | Classical ≤ Prescriptive ≤ DRCC | Classical ≤ Prescriptive ≤ DRCC |
| **Load shedding** | All ≈ 0 (all methods cope) | Classical >> Prescriptive >> DRCC ≈ 0 |
| **Total cost** | All similar | **DRCC wins** — no load shedding |

DRCC pays a small operation cost premium on the close test (conservative policy)
but avoids catastrophic load shedding on the far test. This is the distributional robustness trade-off.


In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.ticker as mticker

methods  = ['Classical', 'Prescriptive', 'Distributionally robust\nprescriptive']
keys_cl  = ['Classical-close', 'Prescriptive-close', 'DRCC-close']
keys_fa  = ['Classical-far',   'Prescriptive-far',   'DRCC-far'  ]
hint_cl  = 'Hindsight-close'
hint_fa  = 'Hindsight-far'

col_m = ['#4878CF', '#E87036', '#CC3333']

def get_vals(keys, field):
    if field == 'op_total':
        return [results[k]['op']+results[k]['on']+results[k]['st'] for k in keys]
    return [results[k][field] for k in keys]

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharey='row')

fields  = ['op_total', 'ls', 'total']
ylabels = ['Production cost ($)', 'Load-shedding cost ($)', 'Total cost ($)']

x3 = np.arange(3); bw = 0.55

col_titles = ['Close test\n(same distribution)',
              'Far test\n(high demand, distribution shift)']
for row, (field, ylabel) in enumerate(zip(fields, ylabels)):
    for col, (keys, hint_key, grp) in enumerate([
            (keys_cl, hint_cl, 'Close test'),
            (keys_fa, hint_fa, 'Far test')]):
        ax = axes[row, col]
        vals = get_vals(keys, field)
        bars = ax.bar(x3, vals, width=bw, color=col_m, edgecolor='k', linewidth=0.8)

        if field == 'op_total':
            hint_val = results[hint_key]['op']+results[hint_key]['on']+results[hint_key]['st']
        else:
            hint_val = results[hint_key][field]
        ax.axhline(hint_val, color='#6ACC65', lw=2, ls='--')

        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height(),
                    f'{v:,.0f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

        ax.set_xticks(x3)
        ax.set_xticklabels(methods, fontsize=10, fontweight='bold')
        ax.set_ylabel(ylabel if col == 0 else '', fontsize=11, fontweight='bold')
        ax.tick_params(axis='y', labelsize=10)
        for tick in ax.get_yticklabels():
            tick.set_fontweight('bold')

        ax.grid(axis='y', alpha=0.35)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
        if row == 0:
            ax.set_title(col_titles[col], fontsize=13, fontweight='bold', pad=10)
            ax.set_ylim(2000, 7200)
        elif row == 2:
            ax.set_ylim(3000, 11500)

      

# Legend in upper-left of first subplot (axes[0, 0])
legend_els = [Patch(fc=c, ec='k', label=m) for c, m in zip(col_m, methods)]
legend_els += [Line2D([0], [0], color='#6ACC65', lw=2, ls='--', label='Hindsight')]

leg = axes[0, 0].legend(handles=legend_els, fontsize=9, frameon=True,
                         loc='upper left', borderpad=0.5, labelspacing=0.4)
for text in leg.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()
plt.savefig('oos_results_1train_2test.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: stacked composition (close vs far side by side for each method) ─
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for col, (keys, hint_key, grp) in enumerate([
        (keys_cl, hint_cl, 'Close Test (same distribution)'),
        (keys_fa, hint_fa, 'Far Test (near-peak, mild distribution shift)')]):
    ax = axes[col]
    op  = get_vals(keys, 'op_total')
    ls  = get_vals(keys, 'ls')

    ax.bar(x3, op, width=bw, color=col_m, edgecolor='k', lw=0.8,
           label='Operation+On+Startup')
    ax.bar(x3, ls, width=bw, bottom=op, color=col_m, edgecolor='k', lw=0.8,
           hatch='xx', alpha=0.55, label='Load shedding')

    for i,(o,l) in enumerate(zip(op,ls)):
        ax.text(i, o+l, f'Total:\n{o+l:,.0f}', ha='center', va='bottom',
                fontsize=8.5, fontweight='bold')

    if field == 'op_total':
        h_op = results[hint_key]['op']+results[hint_key]['on']+results[hint_key]['st']
    else:
        h_op = results[hint_key]['op']+results[hint_key]['on']+results[hint_key]['st']
    h_ls = results[hint_key]['ls']
    ax.axhline(h_op+h_ls, color='#6ACC65', lw=2.5, ls='--',
               label=f'Hindsight ({h_op+h_ls:,.0f})')

    ax.set_xticks(x3); ax.set_xticklabels(methods, fontsize=11)
    ax.set_ylabel('Cost ($)'); ax.grid(axis='y', alpha=0.35)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:,.0f}'))
    ax.set_title(f'{grp}\nSolid = operation  |  Hatched = load shedding',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8.5, loc='upper right')

fig.suptitle('Cost Composition: DRCC hedges against the far-test distribution shift\n'
             'Close test: similar costs  |  Far test: DRCC avoids load-shedding entirely',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('oos_stacked_1train_2test.pdf', bbox_inches='tight')
plt.show()


In [ ]:
print('='*95)
print(f"{'Method':<22} {'Test set':<10} {'Op+On+St':>12} {'Load-shed':>12} {'Total':>12} {'LS%':>8}")
print('-'*95)
for lbl, bv, zt in eval_cases:
    r=results[lbl]
    op=r['op']+r['on']+r['st']; ls=r['ls']; tot=r['total']
    pct=100*ls/tot if tot>0 else 0
    test_tag = 'close' if 'close' in lbl else 'far  '
    meth = lbl.replace('-close','').replace('-far','')
    print(f'{meth:<22} {test_tag:<10} {op:>12,.0f} {ls:>12,.0f} {tot:>12,.0f} {pct:>7.1f}%')
print('='*95)
print()
for split in ['close','far']:
    cl_key = f'Classical-{split}'; dr_key = f'DRCC-{split}'
    print(f'DRCC vs Classical ({split} test):')
    for key in ['op','on','st','ls','total']:
        d = results[dr_key][key] - results[cl_key][key]
        print(f'  Delta {key:8s} = {d:+,.0f}')
    print()
